[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/duckdb-certified/notebooks/day-07-writing-exporting.ipynb#scrollTo=a1b2c3d4)

---
# Day 7 · Writing and Exporting — COPY TO, EXPORT DATABASE, and Parquet Tuning
**certified-journeys / duckdb-certified** &nbsp;|&nbsp; Writing & Exporting

> **Goal for today:** Write query results and full databases to disk in Parquet, CSV, and JSON using DuckDB's COPY TO and EXPORT DATABASE statements — and tune compression settings to minimise output file sizes.

---
## Why exporting matters for analytical engineers

DuckDB is excellent at reading remote data, but equally powerful at writing it. You'll need to export results when:

| Use case | Right tool |
|---|---|
| Share a query result as a file | `COPY (SELECT …) TO 'file.parquet'` |
| Back up or migrate a full database | `EXPORT DATABASE 'dir/'` |
| Produce a downstream Parquet for Spark/Athena | `COPY … (FORMAT PARQUET, COMPRESSION ZSTD)` |
| Generate a CSV for a spreadsheet user | `COPY … (FORMAT CSV, HEADER true, DELIMITER ',')` |
| Restore a backup into a fresh file | `IMPORT DATABASE 'dir/'` |

> **Read first:** [DuckDB COPY Statement docs](https://duckdb.org/docs/sql/statements/copy)

In [ ]:
%pip install -q duckdb

---
## Step 1 · Set up a sample in-memory database

Before exporting anything, we need data. We'll create two tables — `orders` and `products` — and populate them with realistic-looking rows.

In [ ]:
import duckdb
import os, pathlib, json

# In-memory database for our sample data
con = duckdb.connect()

con.execute("""
CREATE TABLE products AS
SELECT
    row_number() OVER ()                                    AS product_id,
    'Product ' || chr(64 + (row_number() OVER () % 26 + 1)) AS name,
    round(random() * 200 + 10, 2)                          AS price,
    CASE (row_number() OVER () % 4)
        WHEN 0 THEN 'Electronics'
        WHEN 1 THEN 'Clothing'
        WHEN 2 THEN 'Books'
        ELSE 'Home'
    END AS category
FROM range(1, 51)  -- 50 products
""")

con.execute("""
CREATE TABLE orders AS
SELECT
    row_number() OVER ()                   AS order_id,
    (random() * 49 + 1)::INT               AS product_id,
    (random() * 9 + 1)::INT                AS quantity,
    current_date - (random() * 365)::INT   AS order_date,
    CASE (row_number() OVER () % 3)
        WHEN 0 THEN 'shipped'
        WHEN 1 THEN 'pending'
        ELSE 'delivered'
    END AS status
FROM range(1, 1001)  -- 1 000 orders
""")

# Quick sanity check
print("products:", con.execute("SELECT count(*) FROM products").fetchone()[0])
print("orders:  ", con.execute("SELECT count(*) FROM orders").fetchone()[0])
print()
print(con.execute("SELECT * FROM orders LIMIT 3").df())

**What just happened?**
- `range(1, 51)` generates 50 rows; `row_number() OVER ()` assigns sequential IDs
- `random()` produces a uniform float in [0, 1) — multiply and cast to INT for realistic ranges
- Both tables live in memory; nothing is on disk yet

---
## Step 2 · COPY (SELECT …) TO — export a query result to Parquet

The most common export pattern: run a query, write its result to a file.

```sql
COPY (SELECT ...) TO 'path/to/file.parquet' (FORMAT PARQUET);
```

Key options for Parquet output:

| Option | Values | Effect |
|---|---|---|
| `FORMAT` | `PARQUET` | Write as Parquet (default for `.parquet` extension) |
| `COMPRESSION` | `SNAPPY`, `ZSTD`, `GZIP`, `NONE` | Codec applied to each column chunk |
| `ROW_GROUP_SIZE` | integer | Rows per row group (default 122 880) |
| `FIELD_IDS` | `auto` | Write Parquet field IDs for better schema compatibility |

In [ ]:
import os

# Create an output directory
os.makedirs('/tmp/duckdb_export', exist_ok=True)

# Export a filtered join to Parquet with default compression (Snappy)
con.execute("""
COPY (
    SELECT
        o.order_id,
        o.order_date,
        o.status,
        p.name        AS product_name,
        p.category,
        p.price,
        o.quantity,
        round(p.price * o.quantity, 2) AS line_total
    FROM orders o
    JOIN products p USING (product_id)
    WHERE o.status = 'delivered'
    ORDER BY o.order_date DESC
)
TO '/tmp/duckdb_export/delivered_orders.parquet'
(FORMAT PARQUET)
""")

size_bytes = os.path.getsize('/tmp/duckdb_export/delivered_orders.parquet')
print(f"Exported: delivered_orders.parquet  ({size_bytes:,} bytes)")

# Verify we can read it back
result = con.execute("""
    SELECT count(*), min(order_date), max(order_date)
    FROM read_parquet('/tmp/duckdb_export/delivered_orders.parquet')
""").fetchone()
print(f"Rows: {result[0]}, Date range: {result[1]} → {result[2]}")

**What just happened?**
- `COPY (SELECT …) TO` runs the query and writes results directly — **no intermediate Python object**
- Only `status = 'delivered'` rows were exported — the filter ran inside DuckDB
- `read_parquet()` can immediately query the file we just wrote
- **Default compression is Snappy** — fast but not the most compact

---
## Step 3 · COPY TO CSV — custom delimiter, header, and quoting

CSV export is frequently needed for stakeholders or legacy systems. DuckDB's CSV writer supports:

| Option | Example | Meaning |
|---|---|---|
| `HEADER` | `true` | Include column names in first row |
| `DELIMITER` | `'\t'` or `';'` | Field separator (default `,`) |
| `QUOTE` | `'"'` | Quote character for string fields |
| `ESCAPE` | `'\\'` | Escape character for quotes inside strings |
| `NULLSTR` | `'NULL'` | String to write for NULL values |
| `DATEFORMAT` | `'%Y-%m-%d'` | strftime-style date format |

In [ ]:
# Export a summary to CSV with custom options
con.execute("""
COPY (
    SELECT
        p.category,
        count(*)                        AS order_count,
        sum(o.quantity)                 AS units_sold,
        round(avg(p.price), 2)          AS avg_price,
        round(sum(p.price * o.quantity), 2) AS total_revenue
    FROM orders o
    JOIN products p USING (product_id)
    GROUP BY p.category
    ORDER BY total_revenue DESC
)
TO '/tmp/duckdb_export/category_summary.csv'
(
    FORMAT     CSV,
    HEADER     true,
    DELIMITER  ',',
    QUOTE      '"',
    NULLSTR    'N/A',
    DATEFORMAT '%Y-%m-%d'
)
""")

# Show the file content
with open('/tmp/duckdb_export/category_summary.csv') as f:
    print(f.read())

# Also export as tab-separated for Excel compatibility
con.execute("""
COPY (
    SELECT order_id, order_date, status, product_id, quantity
    FROM orders
    LIMIT 10
)
TO '/tmp/duckdb_export/orders_sample.tsv'
(
    FORMAT    CSV,
    HEADER    true,
    DELIMITER '\t'
)
""")
print("Tab-separated file written:", os.path.getsize('/tmp/duckdb_export/orders_sample.tsv'), "bytes")

**What just happened?**
- `HEADER true` adds column names as the first line
- `NULLSTR 'N/A'` writes the string `N/A` wherever a SQL NULL would appear
- `DELIMITER '\t'` changes to TSV — useful for Excel's open-file dialog
- **The aggregation ran inside DuckDB** — only the tiny result set was written to disk

---
## Step 4 · EXPORT DATABASE — snapshot a full DuckDB file

When you need to back up or migrate an entire DuckDB database, `EXPORT DATABASE` writes every table as a Parquet file plus a `schema.sql` file that recreates the schema.

```sql
EXPORT DATABASE 'export_dir/' (FORMAT PARQUET, COMPRESSION ZSTD);
```

The output directory contains:
- `schema.sql` — `CREATE TABLE` statements for every table
- One `.parquet` file per table

> **Reference:** [EXPORT DATABASE docs](https://duckdb.org/docs/sql/statements/export)

In [ ]:
import os

export_dir = '/tmp/duckdb_export/full_export'
os.makedirs(export_dir, exist_ok=True)

# Export every table in the database to a directory
con.execute(f"""
EXPORT DATABASE '{export_dir}'
(FORMAT PARQUET, COMPRESSION ZSTD)
""")

# Inspect what was created
exported_files = sorted(os.listdir(export_dir))
print("Files in export directory:")
for fname in exported_files:
    fpath = os.path.join(export_dir, fname)
    size = os.path.getsize(fpath)
    print(f"  {fname:<40}  {size:>8,} bytes")

print()
# Show the schema.sql content
with open(os.path.join(export_dir, 'schema.sql')) as f:
    print("=== schema.sql ===")
    print(f.read())

**What just happened?**
- `EXPORT DATABASE` serialised every table to a separate `.parquet` file
- `schema.sql` contains `CREATE TABLE` statements — the schema is portable
- **ZSTD compression** was applied to all Parquet files — typically 20–30% smaller than Snappy
- The export directory is fully self-contained: Parquet + SQL = everything needed to restore

---
## Step 5 · Parquet tuning — ROW_GROUP_SIZE and COMPRESSION comparison

Parquet stores data in *row groups* — horizontal slices of the table. Each row group is independently compressed.

| Codec | Speed | Size | Best for |
|---|---|---|---|
| `SNAPPY` | Very fast | Medium | Intermediate/temp files, fast reads |
| `ZSTD` | Fast (level 1–3) | Smallest | Production Parquet — best ratio |
| `GZIP` | Slow | Small | Interchange with non-DuckDB tools |
| `NONE` | Fastest | Largest | Debugging, already-compressed data |

**`ROW_GROUP_SIZE`** controls how many rows per group. Smaller groups = faster predicate pushdown for selective queries. Larger groups = better compression ratio.

In [ ]:
import os

# Export the same 1 000 orders with different compression codecs and row group sizes
configs = [
    ('snappy_default',  'SNAPPY', 122880),
    ('zstd_default',    'ZSTD',   122880),
    ('zstd_small_rg',   'ZSTD',   100),
    ('gzip_default',    'GZIP',   122880),
    ('none_uncompressed', 'NONE', 122880),
]

results = []
for label, codec, rg_size in configs:
    path = f'/tmp/duckdb_export/orders_{label}.parquet'
    con.execute(f"""
    COPY (
        SELECT o.*, p.name AS product_name, p.category, p.price
        FROM orders o JOIN products p USING (product_id)
    )
    TO '{path}'
    (FORMAT PARQUET, COMPRESSION {codec}, ROW_GROUP_SIZE {rg_size})
    """)
    size = os.path.getsize(path)
    results.append((label, codec, rg_size, size))

# Print comparison table
baseline = results[0][3]  # Snappy is our baseline
print(f"{'Label':<25} {'Codec':<8} {'RowGroup':>10} {'Bytes':>10} {'vs Snappy':>12}")
print('-' * 68)
for label, codec, rg, size in results:
    pct = (size - baseline) / baseline * 100
    sign = '+' if pct > 0 else ''
    print(f"{label:<25} {codec:<8} {rg:>10,} {size:>10,}  {sign}{pct:>+.1f}%")

**What just happened?**
- ZSTD typically beats Snappy by 15–30% on analytical data with repeated values
- A tiny `ROW_GROUP_SIZE` (100) inflates the file because metadata overhead dominates
- NONE is largest — raw column data with no compression
- **DuckDB's vectorised Parquet reader handles ZSTD at near-Snappy speed** for sequential scans

---
## Step 6 · IMPORT DATABASE — restore from an export snapshot

`IMPORT DATABASE` is the inverse of `EXPORT DATABASE`. It reads `schema.sql`, creates all tables, then bulk-loads the Parquet files.

```sql
IMPORT DATABASE 'export_dir/';
```

This is how you:
- Migrate a DuckDB file to a new machine
- Restore a backup after an accidental drop
- Clone a database for a colleague

In [ ]:
import duckdb, os

restored_db_path = '/tmp/duckdb_export/restored.duckdb'
# Remove any leftover file from a prior run
if os.path.exists(restored_db_path):
    os.remove(restored_db_path)

# Open a fresh persistent DuckDB file — completely empty
restored = duckdb.connect(restored_db_path)

# Import from the export directory we created in Step 4
export_dir = '/tmp/duckdb_export/full_export'
restored.execute(f"IMPORT DATABASE '{export_dir}'")

# Verify row counts match the original
tables = ['orders', 'products']
print("Row count comparison (original → restored):")
for tbl in tables:
    original_count = con.execute(f"SELECT count(*) FROM {tbl}").fetchone()[0]
    restored_count  = restored.execute(f"SELECT count(*) FROM {tbl}").fetchone()[0]
    match = '✓' if original_count == restored_count else '✗ MISMATCH'
    print(f"  {tbl:<12}  original={original_count:>5}  restored={restored_count:>5}  {match}")

# Spot-check a row value
orig_row  = con.execute("SELECT * FROM orders WHERE order_id = 1").fetchone()
rest_row  = restored.execute("SELECT * FROM orders WHERE order_id = 1").fetchone()
print(f"\nOrder #1 original:  {orig_row}")
print(f"Order #1 restored:  {rest_row}")
print("Match:", orig_row == rest_row)

restored.close()

**What just happened?**
- A fresh, empty DuckDB file was opened — no tables existed yet
- `IMPORT DATABASE` read `schema.sql` first, then bulk-loaded each Parquet file
- Row counts and values match exactly — the round-trip is lossless
- **This is the DuckDB backup/restore pattern** for file-based deployments

---
## Step 7 · Inspecting Parquet metadata with parquet_schema() and parquet_metadata()

DuckDB exposes two handy functions for interrogating Parquet files without loading all the data:

| Function | Returns |
|---|---|
| `parquet_schema('file.parquet')` | Column names, types, field IDs |
| `parquet_metadata('file.parquet')` | Row group stats, compression, row counts |

In [ ]:
import duckdb

# Inspect the schema of the file we exported in Step 2
parquet_file = '/tmp/duckdb_export/delivered_orders.parquet'

print("=== parquet_schema ===")
schema_df = con.execute(f"""
    SELECT file_name, name, type, converted_type
    FROM parquet_schema('{parquet_file}')
""").df()
print(schema_df.to_string(index=False))

print("\n=== parquet_metadata (row group summary) ===")
meta_df = con.execute(f"""
    SELECT
        row_group_id,
        row_group_num_rows,
        row_group_bytes,
        file_offset
    FROM parquet_metadata('{parquet_file}')
    LIMIT 5
""").df()
print(meta_df.to_string(index=False))

**What just happened?**
- `parquet_schema()` reveals column types exactly as written — useful for debugging schema mismatches
- `parquet_metadata()` shows per-row-group byte sizes — tells you if row groups are balanced
- **No rows are scanned** — this reads only the Parquet footer metadata
- These functions work on remote files too: `parquet_schema('s3://bucket/file.parquet')`

---
## Step 8 · Challenge — export a filtered summary with ZSTD and verify it

Apply everything from today to a realistic export pipeline.

In [ ]:
# Challenge:
# 1. Write a COPY statement that exports a monthly revenue summary to
#    '/tmp/duckdb_export/monthly_revenue.parquet' with ZSTD compression
#    and ROW_GROUP_SIZE 500.
#
# The summary should have columns:
#   year_month (formatted as 'YYYY-MM'),
#   category,
#   order_count,
#   total_revenue
#
# 2. After exporting, verify:
#    - The file exists and is smaller than the uncompressed version
#    - Read it back and confirm total_revenue sums to the same value as
#      the original orders + products join

# --- Your solution here ---

# Hint: strftime('%Y-%m', order_date) formats a date column as 'YYYY-MM'
# Hint: parquet_schema() can confirm your column names after the fact


---
## Day 7 key concepts recap

| Concept | What to remember |
|---|---|
| `COPY (SELECT …) TO` | Writes query results to disk — Parquet, CSV, JSON, TSV |
| `FORMAT PARQUET` | Columnar, typed, compressed — best for analytics |
| `COMPRESSION ZSTD` | 20–30% smaller than Snappy at similar read speed |
| `ROW_GROUP_SIZE` | Larger = better compression; smaller = faster selective queries |
| `EXPORT DATABASE` | Snapshots every table to Parquet + `schema.sql` |
| `IMPORT DATABASE` | Restores a full snapshot into a fresh DuckDB file |
| `parquet_schema()` | Reads column types from Parquet footer — no data scan |
| CSV `HEADER / DELIMITER` | Control quoting, separator, and NULL representation |

> **Tip:** Prefer ZSTD compression for Parquet files you'll read back with DuckDB — it's typically 20–30% smaller than Snappy and DuckDB's vectorised reader handles it at nearly the same speed.

---
## What's next
**Day 8** → Performance and query profiling: EXPLAIN, EXPLAIN ANALYZE, JSON profiling output, and tuning slow queries.

Mark Day 7 complete in your [tracker](../index.html).